# Baseline vs Thompson Sampling — Bandit Adaptativo

Datathon 7MLET — Grupo XX

Este notebook implementa:
1. **Baseline determinístico**: sempre recomenda o braço (oferta) com maior taxa histórica de conversão.
2. **Thompson Sampling**: exploração bayesiana com priors Beta por braço.
3. Comparação de performance (conversão acumulada).
4. Golden Set (5 clientes) com a recomendação do modelo.
5. Registro de parâmetros e métricas no **MLflow**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow

np.random.seed(42)
df = pd.read_csv('../data/bank_marketing_arms.csv')

ARMS = ['Oferta_A', 'Oferta_B', 'Oferta_C']
reward_cols = [f'reward_{a}' for a in ARMS]

# embaralha as linhas para simular a chegada sequencial de clientes ("rodadas" do bandit)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
n_rounds = len(df)
print(f"Total de rodadas (clientes): {n_rounds}")
df[reward_cols].mean()

## 1. Baseline determinístico

A política de baseline observa as primeiras N rodadas para descobrir a "melhor oferta histórica" e depois sempre recomenda esse mesmo braço para todo mundo (é o que uma regra fixa faria).

In [ ]:
WARMUP = 200  # rodadas de observação inicial para descobrir o "melhor braço histórico"

warmup_means = df.iloc[:WARMUP][reward_cols].mean()
best_arm_baseline = warmup_means.idxmax().replace('reward_', '')
print(f"Melhor braço identificado no warmup: {best_arm_baseline}")

# baseline aplica esse braço fixo do WARMUP em diante
baseline_rewards = df.iloc[WARMUP:][f'reward_{best_arm_baseline}'].values
baseline_conversion = baseline_rewards.mean()
print(f"Taxa de conversão do Baseline (regra fixa): {baseline_conversion:.4f}")

## 2. Thompson Sampling

Para cada braço mantemos uma distribuição **Beta(alpha, beta)**, prior não-informativo Beta(1,1) (equivale a incerteza total). A cada rodada:
1. Amostramos um valor da Beta de cada braço.
2. Escolhemos o braço com maior amostra (exploração bayesiana).
3. Observamos a recompensa real (conversão) e atualizamos alpha/beta do braço escolhido.

In [ ]:
class ThompsonSampling:
    def __init__(self, arms, alpha_prior=1.0, beta_prior=1.0):
        self.arms = arms
        self.alpha = {a: alpha_prior for a in arms}
        self.beta = {a: beta_prior for a in arms}

    def select_arm(self):
        samples = {a: np.random.beta(self.alpha[a], self.beta[a]) for a in self.arms}
        return max(samples, key=samples.get)

    def update(self, arm, reward):
        if reward == 1:
            self.alpha[arm] += 1
        else:
            self.beta[arm] += 1

ts = ThompsonSampling(ARMS)
ts_rewards = []
ts_chosen_arms = []

eval_df = df.iloc[WARMUP:].reset_index(drop=True)

for i, row in eval_df.iterrows():
    arm = ts.select_arm()
    reward = row[f'reward_{arm}']
    ts.update(arm, reward)
    ts_rewards.append(reward)
    ts_chosen_arms.append(arm)

ts_conversion = np.mean(ts_rewards)
print(f"Taxa de conversão do Thompson Sampling: {ts_conversion:.4f}")
print(f"Ganho sobre o baseline: {(ts_conversion - baseline_conversion) / baseline_conversion * 100:.1f}%")

In [ ]:
from collections import Counter
print("Distribuição de braços escolhidos pelo Thompson Sampling:")
print(Counter(ts_chosen_arms))

## 3. Comparação — conversão acumulada

In [ ]:
baseline_cum = np.cumsum(baseline_rewards) / (np.arange(len(baseline_rewards)) + 1)
ts_cum = np.cumsum(ts_rewards) / (np.arange(len(ts_rewards)) + 1)

plt.figure(figsize=(9,5))
plt.plot(baseline_cum, label=f'Baseline (regra fixa: {best_arm_baseline})', color='#888888')
plt.plot(ts_cum, label='Thompson Sampling', color='#2ecc71')
plt.xlabel('Rodada (cliente atendido)')
plt.ylabel('Taxa de conversão acumulada')
plt.title('Baseline vs Thompson Sampling')
plt.legend()
plt.tight_layout()
plt.savefig('../data/comparacao_baseline_ts.png', dpi=120)
plt.show()

## 4. Golden Set (5 clientes de exemplo)

Selecionamos 5 clientes do dataset e mostramos qual oferta o Thompson Sampling recomendaria para cada um, junto com uma leitura rápida se a decisão faz sentido.

In [ ]:
golden_set = df.sample(5, random_state=7).reset_index(drop=True)

resultados = []
for i, row in golden_set.iterrows():
    samples = {a: np.random.beta(ts.alpha[a], ts.beta[a]) for a in ARMS}
    recomendado = max(samples, key=samples.get)
    resultados.append({
        'cliente_idx': i,
        'idade_padronizada': round(row['age'], 2),
        'braco_recomendado': recomendado,
        'reward_real_no_braco': row[f'reward_{recomendado}']
    })

golden_df = pd.DataFrame(resultados)
golden_df

**Leitura:** o modelo já converge para os braços com maior recompensa histórica (Oferta_C, que tem o fator de resposta mais alto), mas ainda mantém exploração residual nos demais — comportamento esperado de um bandit bayesiano, diferente do baseline que ficaria travado em uma única oferta desde o início.

## 5. Registro no MLflow (Etapa 7)

Registra parâmetros do algoritmo e as métricas de conversão do baseline e do bandit.

In [ ]:
mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("datathon-bandit-ofertas")

with mlflow.start_run(run_name="thompson_sampling_vs_baseline"):
    mlflow.log_param("algoritmo", "thompson_sampling")
    mlflow.log_param("arms", ARMS)
    mlflow.log_param("prior_alpha", 1.0)
    mlflow.log_param("prior_beta", 1.0)
    mlflow.log_param("warmup_rounds", WARMUP)
    mlflow.log_param("dataset", "bank-marketing (Kaggle - henriqueyamahata)")

    mlflow.log_metric("taxa_conversao_baseline", baseline_conversion)
    mlflow.log_metric("taxa_conversao_thompson_sampling", ts_conversion)
    mlflow.log_metric("ganho_percentual", (ts_conversion - baseline_conversion) / baseline_conversion * 100)

    mlflow.log_artifact("../data/comparacao_baseline_ts.png")

print("Run registrada no MLflow. Rode `mlflow ui --backend-store-uri file:../mlruns` para visualizar.")

In [ ]:
import pickle
with open('../data/thompson_model.pkl', 'wb') as f:
    pickle.dump({'alpha': ts.alpha, 'beta': ts.beta, 'arms': ARMS}, f)
print("Estado do modelo (alpha/beta por braço) salvo para uso na API (Etapa 5).")